# Notebook 01: End-to-End Nowcasting Pipeline

**Module 08 — Production Systems**  
**Estimated time**: 15 minutes

## Learning Objectives

By the end of this notebook you will:

1. Implement the five-layer pipeline architecture end-to-end on synthetic macro data
2. Build and query a SQLite vintage database
3. Simulate ragged-edge data using publication lag offsets
4. Generate a sequence of nowcast records for a single target quarter
5. Plot the nowcast evolution chart

## Prerequisites

- Module 07 Notebook 01 (simplified NY Fed nowcast)
- Module 08 Guide 01 (pipeline architecture)

## Setup

We use only the standard scientific Python stack. All data are generated synthetically so no internet access is required.

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
import datetime
import json
import os
import warnings
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Setup complete')

## Part 1: Synthetic Macro Data Generation

We generate 10 years of monthly macro indicators and quarterly GDP growth. The indicators share a common business cycle factor plus idiosyncratic noise. This creates realistic correlations between indicators.

We also generate revision noise: the "preliminary" vintage of each indicator has additional noise that disappears in the "revised" vintage.

In [ ]:
def generate_macro_data(
    n_months: int = 120,
    start_year: int = 2014,
    revision_noise_std: float = 0.3,
    seed: int = 42,
) -> Dict:
    """
    Generate synthetic macro data with:
    - A latent business cycle factor (AR1)
    - 5 monthly indicators loading on the factor
    - Quarterly GDP growth driven by the factor
    - Preliminary vs revised vintages

    Returns dict with keys: dates_monthly, dates_quarterly,
                             indicators (dict), gdp_advance, gdp_revised
    """
    rng = np.random.default_rng(seed)

    # Monthly dates
    dates_m = pd.date_range(
        f'{start_year}-01-01', periods=n_months, freq='MS'
    )

    # Latent business cycle factor (AR1 with rho=0.8)
    rho = 0.80
    factor = np.zeros(n_months)
    for t in range(1, n_months):
        factor[t] = rho * factor[t-1] + rng.normal(0, 0.5)

    # Recession dummy (factor < -1.0 for 2+ consecutive months)
    in_recession = np.zeros(n_months, dtype=bool)
    consec = 0
    for t in range(n_months):
        if factor[t] < -1.0:
            consec += 1
        else:
            consec = 0
        if consec >= 2:
            in_recession[t] = True

    # Five monthly indicators
    # loadings: how strongly each indicator reflects the factor
    loadings = [0.8, 0.6, 0.9, 0.5, 0.7]
    noise_std = [0.4, 0.6, 0.3, 0.8, 0.5]
    pub_lags = [4, 16, 1, 14, 5]  # days after month-end
    names = ['PAYEMS', 'INDPRO', 'ISM_PMI', 'RETAILSL', 'CLAIMS']

    indicators_prelim = {}
    indicators_revised = {}

    for i, name in enumerate(names):
        # Revised (true) values
        revised = loadings[i] * factor + rng.normal(0, noise_std[i], n_months)
        # Preliminary = revised + revision noise
        prelim = revised + rng.normal(0, revision_noise_std, n_months)
        indicators_prelim[name] = pd.Series(prelim, index=dates_m, name=name)
        indicators_revised[name] = pd.Series(revised, index=dates_m, name=name)

    # Quarterly GDP growth from the factor (3-month average)
    n_quarters = n_months // 3
    dates_q = pd.date_range(
        f'{start_year}-01-01', periods=n_quarters, freq='QS'
    )
    gdp = np.zeros(n_quarters)
    for q in range(n_quarters):
        m_start = q * 3
        gdp[q] = (
            0.5 * factor[m_start : m_start+3].mean()
            + rng.normal(0, 0.3)
        )

    gdp_advance = pd.Series(gdp + rng.normal(0, 0.1, n_quarters), index=dates_q)
    gdp_revised = pd.Series(gdp, index=dates_q)

    return {
        'dates_monthly': dates_m,
        'dates_quarterly': dates_q,
        'indicators_prelim': indicators_prelim,
        'indicators_revised': indicators_revised,
        'gdp_advance': gdp_advance,
        'gdp_revised': gdp_revised,
        'pub_lags': dict(zip(names, pub_lags)),
        'factor': factor,
    }

macro = generate_macro_data(n_months=120)
print('Macro data generated')
print(f"Monthly obs: {len(macro['dates_monthly'])}")
print(f"Quarterly obs: {len(macro['dates_quarterly'])}")
print(f"Indicators: {list(macro['indicators_prelim'].keys())}")

## Part 2: Vintage Database

We build a SQLite vintage database and populate it with two vintages for each indicator:

- **Preliminary vintage**: released `pub_lag` days after the reference month ends
- **Revised vintage**: released one month later (subsequent preliminary release date)

In [ ]:
class VintageDatabase:
    """SQLite-backed vintage database."""

    def __init__(self, db_path: str = ':memory:'):
        self.conn = sqlite3.connect(db_path)
        self._create_tables()

    def _create_tables(self) -> None:
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS vintages (
                series_id    TEXT NOT NULL,
                obs_date     TEXT NOT NULL,
                vintage_date TEXT NOT NULL,
                value        REAL,
                PRIMARY KEY (series_id, obs_date, vintage_date)
            )
        """)
        self.conn.commit()

    def store_bulk(self, series_id: str, data: pd.Series, vintage_date: str) -> None:
        rows = [
            (series_id, str(idx.date()), vintage_date, float(val))
            for idx, val in data.items() if pd.notna(val)
        ]
        self.conn.executemany(
            'INSERT OR REPLACE INTO vintages VALUES (?, ?, ?, ?)', rows
        )
        self.conn.commit()

    def get_as_of(self, series_id: str, as_of_date: str) -> pd.Series:
        """Return series using latest vintage available as of as_of_date."""
        df = pd.read_sql_query(
            """
            SELECT obs_date, value FROM vintages
            WHERE series_id = ? AND vintage_date <= ?
            ORDER BY obs_date, vintage_date DESC
            """,
            self.conn,
            params=(series_id, as_of_date),
        )
        if df.empty:
            return pd.Series(dtype=float, name=series_id)
        df = df.drop_duplicates(subset='obs_date', keep='first')
        df['obs_date'] = pd.to_datetime(df['obs_date'])
        return df.set_index('obs_date')['value'].sort_index()


def populate_vintage_db(macro_data: Dict) -> VintageDatabase:
    """
    Populate a vintage database with preliminary and revised vintages.
    Preliminary released pub_lag days after month-end.
    Revised released one month later.
    """
    db = VintageDatabase()

    for name, series in macro_data['indicators_prelim'].items():
        lag = macro_data['pub_lags'][name]
        for obs_date, val in series.items():
            # Preliminary: stored on month-end + lag days
            month_end = obs_date + pd.offsets.MonthEnd(0)
            prelim_date = (month_end + datetime.timedelta(days=lag)).strftime('%Y-%m-%d')
            db.store_bulk(
                name,
                pd.Series([val], index=[obs_date], name=name),
                prelim_date,
            )

    for name, series in macro_data['indicators_revised'].items():
        lag = macro_data['pub_lags'][name]
        for obs_date, val in series.items():
            month_end = obs_date + pd.offsets.MonthEnd(0)
            # Revised: one month after the preliminary
            revised_date = (
                month_end
                + pd.DateOffset(months=1)
                + datetime.timedelta(days=lag)
            ).strftime('%Y-%m-%d')
            db.store_bulk(
                name,
                pd.Series([val], index=[obs_date], name=name),
                revised_date,
            )

    return db


vintage_db = populate_vintage_db(macro)
print('Vintage database populated')

# Test: what did PAYEMS look like on 2022-11-15?
sample = vintage_db.get_as_of('PAYEMS', '2022-11-15')
print(f'PAYEMS as of 2022-11-15: {len(sample)} observations, last = {sample.index[-1].date()}')

## Part 3: Feature Engineering — Ragged-Edge Fill and MIDAS Matrix

Given a forecast date (the `as_of_date`), we:
1. Query each series as of that date
2. Fill missing periods up to the current month using carry-forward
3. Build the MIDAS lag matrix (3 lags per indicator)

In [ ]:
def fill_ragged_edge(series: pd.Series, target_end: pd.Timestamp) -> pd.Series:
    """
    Extend series to target_end using carry-forward fill.
    Only adds periods that are missing; never shortens the series.
    """
    if series.empty:
        return series
    full_idx = pd.date_range(series.index[0], target_end, freq='MS')
    return series.reindex(full_idx).ffill()


def build_midas_features(
    vintage_db: VintageDatabase,
    series_names: List[str],
    quarterly_dates: pd.DatetimeIndex,
    as_of_date: str,
    n_lags: int = 3,
) -> Tuple[np.ndarray, List[str]]:
    """
    Build the MIDAS feature matrix for a set of quarterly target dates.

    For each quarterly date, we take the n_lags most recent monthly
    observations of each indicator available as of as_of_date.

    Returns
    -------
    X : np.ndarray, shape (n_quarters, n_lags * n_series)
    feature_names : list of strings
    """
    current_month = pd.Timestamp(as_of_date).to_period('M').to_timestamp()

    # Fetch and fill each series
    series_data = {}
    for name in series_names:
        s = vintage_db.get_as_of(name, as_of_date)
        s = fill_ragged_edge(s, current_month)
        series_data[name] = s

    n_quarters = len(quarterly_dates)
    n_features = n_lags * len(series_names)
    X = np.full((n_quarters, n_features), np.nan)
    feature_names = []

    col = 0
    for name in series_names:
        s = series_data[name]
        for lag_idx in range(n_lags):
            feature_names.append(f'{name}_lag{lag_idx+1}')
            for t, q_date in enumerate(quarterly_dates):
                # Monthly data available through end of the quarter
                # lag 1 = most recent month of the quarter
                q_month = q_date + pd.DateOffset(months=2)  # end month of quarter
                target_month = q_month - pd.DateOffset(months=lag_idx)
                if target_month in s.index:
                    X[t, col] = s[target_month]
            col += 1

    return X, feature_names


# Test feature construction for Q4 2022 target
q_dates = macro['dates_quarterly'][:28]  # train window
X_test, feat_names = build_midas_features(
    vintage_db,
    list(macro['indicators_prelim'].keys()),
    q_dates,
    as_of_date='2022-11-15',
    n_lags=3,
)
print(f'Feature matrix shape: {X_test.shape}')
print(f'Feature names (first 6): {feat_names[:6]}')
nan_frac = np.isnan(X_test).mean()
print(f'NaN fraction: {nan_frac:.2%}')

## Part 4: The Pipeline Orchestrator

We implement a simplified version of the five-layer architecture. Each pipeline run:
1. Determines the training window (all quarters with complete data before `as_of_date`)
2. Builds the MIDAS feature matrix
3. Fits ElasticNetCV with TimeSeriesSplit
4. Predicts the target quarter
5. Returns a `ForecastRecord`

In [ ]:
@dataclass
class ForecastRecord:
    forecast_date: str
    target_quarter: str
    point_forecast: float
    lower_95: float
    upper_95: float
    n_train: int
    model: str


def run_pipeline(
    vintage_db: VintageDatabase,
    gdp_series: pd.Series,
    series_names: List[str],
    all_quarterly_dates: pd.DatetimeIndex,
    as_of_date: str,
    target_quarter: pd.Timestamp,
    n_lags: int = 3,
    model_type: str = 'elasticnet',
) -> Optional[ForecastRecord]:
    """
    Run the nowcasting pipeline for a single (as_of_date, target_quarter) pair.

    Training window: all quarters strictly before target_quarter with GDP known.
    Features: built using vintage data as of as_of_date.
    """
    # Training quarters: before target_quarter and GDP available
    train_mask = all_quarterly_dates < target_quarter
    train_dates = all_quarterly_dates[train_mask]

    if len(train_dates) < 20:
        return None  # insufficient history

    # Build feature matrix for training + forecast quarters
    all_dates = pd.DatetimeIndex(list(train_dates) + [target_quarter])
    X, _ = build_midas_features(
        vintage_db, series_names, all_dates, as_of_date, n_lags=n_lags
    )

    X_train = X[:-1]  # all but last
    X_forecast = X[-1:]  # target quarter
    y_train = gdp_series[train_dates].values

    # Drop rows with NaN
    mask = ~np.isnan(X_train).any(axis=1) & ~np.isnan(y_train)
    X_train = X_train[mask]
    y_train = y_train[mask]

    if len(y_train) < 16 or np.isnan(X_forecast).any():
        return None

    # Scale
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train)
    X_fc_s = scaler.transform(X_forecast)

    # Fit model
    cv = TimeSeriesSplit(n_splits=5)
    if model_type == 'elasticnet':
        model = ElasticNetCV(
            l1_ratio=[0.5, 0.7, 1.0],
            cv=cv, max_iter=5000, random_state=0
        )
    else:
        model = RidgeCV(cv=cv)

    model.fit(X_tr_s, y_train)
    point = float(model.predict(X_fc_s)[0])

    # Parametric prediction interval
    resid = y_train - model.predict(X_tr_s)
    sigma = resid.std()
    n = len(y_train)
    t95 = stats.t.ppf(0.975, df=n - 2)

    return ForecastRecord(
        forecast_date=as_of_date,
        target_quarter=str(target_quarter.date()),
        point_forecast=point,
        lower_95=point - t95 * sigma,
        upper_95=point + t95 * sigma,
        n_train=n,
        model=model_type,
    )


# Test a single pipeline run
target_q = pd.Timestamp('2022-10-01')  # Q4 2022
record = run_pipeline(
    vintage_db,
    macro['gdp_advance'],
    list(macro['indicators_prelim'].keys()),
    macro['dates_quarterly'],
    as_of_date='2022-11-15',
    target_quarter=target_q,
)
if record:
    print(f'Point forecast: {record.point_forecast:.3f}')
    print(f'95% interval: [{record.lower_95:.3f}, {record.upper_95:.3f}]')
    print(f'Training obs: {record.n_train}')

## Part 5: Nowcast Evolution — Simulating Multiple Pipeline Runs

We simulate 6 pipeline runs for Q4 2022, starting when the first data become available (October PMI, released November 1) through the week before the advance GDP release.

Each run uses a different `as_of_date`, so different series are available at each step — this is the ragged-edge structure in action.

In [ ]:
# Simulate pipeline runs throughout Q4 2022 nowcasting period
# Each as_of_date corresponds to a data release event
TARGET_QUARTER = pd.Timestamp('2022-10-01')  # Q4 2022

forecast_dates = [
    '2022-10-04',   # Payrolls for September (pub lag 4 days after Sep 30)
    '2022-10-17',   # CPI and IP for September
    '2022-11-04',   # Payrolls for October
    '2022-11-15',   # Mid-month: claims + other indicators
    '2022-12-02',   # Payrolls for November
    '2022-12-16',   # IP for November (close to end of quarter)
]

release_labels = [
    'Sep Payrolls',
    'Sep CPI/IP',
    'Oct Payrolls',
    'Mid-Nov Claims',
    'Nov Payrolls',
    'Nov IP',
]

evolution_records = []
for fd in forecast_dates:
    rec = run_pipeline(
        vintage_db,
        macro['gdp_advance'],
        list(macro['indicators_prelim'].keys()),
        macro['dates_quarterly'],
        as_of_date=fd,
        target_quarter=TARGET_QUARTER,
    )
    if rec:
        evolution_records.append(rec)
        print(f"{fd}: forecast = {rec.point_forecast:.3f} "
              f"[{rec.lower_95:.3f}, {rec.upper_95:.3f}]")
    else:
        print(f'{fd}: pipeline returned None (insufficient data)')

actual_q4_2022 = float(macro['gdp_advance'][TARGET_QUARTER])
print(f'\nActual Q4 2022 GDP (advance): {actual_q4_2022:.3f}')

## Part 6: Nowcast Evolution Chart

Plot the sequence of nowcasts alongside the actual GDP release.

In [ ]:
def plot_nowcast_evolution(
    records: List[ForecastRecord],
    actual: float,
    labels: List[str],
    target: str,
) -> None:
    if not records:
        print('No records to plot')
        return

    dates = pd.to_datetime([r.forecast_date for r in records])
    points = [r.point_forecast for r in records]
    lower = [r.lower_95 for r in records]
    upper = [r.upper_95 for r in records]

    fig, ax = plt.subplots(figsize=(11, 5))

    ax.fill_between(dates, lower, upper, alpha=0.15, color='steelblue',
                    label='95% prediction interval')
    ax.plot(dates, points, color='steelblue', linewidth=2.5,
            marker='o', markersize=7, label='Nowcast')

    ax.axhline(actual, color='firebrick', linewidth=1.5, linestyle='--',
               label=f'Advance GDP: {actual:.2f}')

    for i, (d, lbl) in enumerate(zip(dates, labels)):
        ax.annotate(
            lbl,
            xy=(d, points[i]),
            xytext=(0, 14),
            textcoords='offset points',
            ha='center',
            fontsize=7.5,
            color='steelblue',
        )

    ax.set_title(
        f'Nowcast Evolution — {target} GDP Growth',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlabel('Forecast date')
    ax.set_ylabel('GDP growth (standardised units)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.xticks(rotation=25, ha='right')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_nowcast_evolution(
    evolution_records,
    actual_q4_2022,
    release_labels[:len(evolution_records)],
    target='Q4 2022',
)

## Part 7: Expanding-Window OOS Evaluation

We run a full pseudo-real-time evaluation across all quarters in the second half of the sample. For each target quarter, we use the terminal forecast (the run closest to the advance GDP release date).

In [ ]:
def pseudo_realtime_eval(
    vintage_db: VintageDatabase,
    gdp_series: pd.Series,
    series_names: List[str],
    all_quarterly_dates: pd.DatetimeIndex,
    eval_quarters: pd.DatetimeIndex,
    n_lags: int = 3,
) -> pd.DataFrame:
    """
    Pseudo-real-time evaluation: for each eval quarter, run the pipeline
    at the terminal forecast date (15 days before quarter-end advance release).
    """
    results = []
    ar1_preds = []

    for tq in eval_quarters:
        # Terminal forecast date: end of the quarter (3 months in)
        # Advance GDP released ~28 days after quarter-end
        # Use 15 days before the advance release as terminal date
        quarter_end = tq + pd.offsets.QuarterEnd(0)
        terminal_date = (quarter_end + datetime.timedelta(days=13)).strftime('%Y-%m-%d')

        rec = run_pipeline(
            vintage_db,
            gdp_series,
            series_names,
            all_quarterly_dates,
            as_of_date=terminal_date,
            target_quarter=tq,
            n_lags=n_lags,
        )

        actual = float(gdp_series.get(tq, np.nan))

        # AR(1) benchmark: last known GDP value
        past = gdp_series[all_quarterly_dates < tq].dropna()
        ar1_pred = float(past.iloc[-1]) if len(past) >= 1 else np.nan

        if rec and not np.isnan(actual):
            results.append({
                'quarter': str(tq.date()),
                'forecast': rec.point_forecast,
                'actual': actual,
                'ar1': ar1_pred,
                'error_midas': rec.point_forecast - actual,
                'error_ar1': ar1_pred - actual,
            })

    return pd.DataFrame(results)


# Evaluate on last 8 quarters of the sample
n_q = len(macro['dates_quarterly'])
eval_quarters = macro['dates_quarterly'][n_q - 8 : n_q - 1]

eval_df = pseudo_realtime_eval(
    vintage_db,
    macro['gdp_advance'],
    list(macro['indicators_prelim'].keys()),
    macro['dates_quarterly'],
    eval_quarters,
)

if len(eval_df) > 0:
    rmse_midas = np.sqrt((eval_df['error_midas'] ** 2).mean())
    rmse_ar1 = np.sqrt((eval_df['error_ar1'] ** 2).mean())
    bias = eval_df['error_midas'].mean()

    print(f'Pseudo-real-time evaluation ({len(eval_df)} quarters)')
    print(f'  MIDAS RMSE  : {rmse_midas:.4f}')
    print(f'  AR(1) RMSE  : {rmse_ar1:.4f}')
    print(f'  RMSE ratio  : {rmse_midas / rmse_ar1:.3f}')
    print(f'  Bias        : {bias:+.4f}')
else:
    print('No evaluation results — increase sample size')

## Part 8: News Decomposition

For linear models (Ridge, ElasticNet), the nowcast revision between two pipeline runs decomposes as:

$$\Delta \hat{y} = \sum_i \hat{\beta}_i \cdot (x_i^{\text{new}} - x_i^{\text{old}})$$

We compute this by comparing the feature vectors from two consecutive runs.

In [ ]:
def compute_news_decomposition(
    vintage_db: VintageDatabase,
    gdp_series: pd.Series,
    series_names: List[str],
    all_quarterly_dates: pd.DatetimeIndex,
    date_prev: str,
    date_curr: str,
    target_quarter: pd.Timestamp,
    n_lags: int = 3,
) -> Dict:
    """
    Decompose the nowcast revision between date_prev and date_curr.

    Uses the model coefficients from date_curr and computes the
    feature change for each indicator group.
    """
    train_mask = all_quarterly_dates < target_quarter
    train_dates = all_quarterly_dates[train_mask]

    if len(train_dates) < 20:
        return {}

    all_dates = pd.DatetimeIndex(list(train_dates) + [target_quarter])

    # Build features at both dates
    X_prev, feat_names = build_midas_features(
        vintage_db, series_names, all_dates, date_prev, n_lags=n_lags
    )
    X_curr, _ = build_midas_features(
        vintage_db, series_names, all_dates, date_curr, n_lags=n_lags
    )

    X_train = X_curr[:-1]
    y_train = gdp_series[train_dates].values

    mask = ~np.isnan(X_train).any(axis=1) & ~np.isnan(y_train)
    X_train = X_train[mask]
    y_train = y_train[mask]

    if len(y_train) < 16:
        return {}

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train)

    cv = TimeSeriesSplit(n_splits=5)
    model = ElasticNetCV(
        l1_ratio=[0.5, 1.0], cv=cv, max_iter=5000, random_state=0
    )
    model.fit(X_tr_s, y_train)

    # Feature vectors for the target quarter
    x_prev = X_prev[-1].copy()
    x_curr = X_curr[-1].copy()

    # Scale both using the same scaler
    x_prev_s = scaler.transform(x_prev.reshape(1, -1))[0]
    x_curr_s = scaler.transform(x_curr.reshape(1, -1))[0]

    # News = coeff * feature change
    coefs = model.coef_
    delta_x = x_curr_s - x_prev_s
    contributions = coefs * delta_x

    # Aggregate by indicator
    news = {}
    for j, name in enumerate(series_names):
        start = j * n_lags
        end = (j + 1) * n_lags
        news[name] = float(contributions[start:end].sum())

    return news


# Compute news for the revision from Nov 4 to Nov 15
news = compute_news_decomposition(
    vintage_db,
    macro['gdp_advance'],
    list(macro['indicators_prelim'].keys()),
    macro['dates_quarterly'],
    date_prev='2022-11-04',
    date_curr='2022-11-15',
    target_quarter=TARGET_QUARTER,
)

print('News decomposition (Nov 4 → Nov 15):')
for k, v in sorted(news.items(), key=lambda x: abs(x[1]), reverse=True):
    bar = '█' * int(abs(v) * 20) if abs(v) > 0 else ''
    sign = '+' if v >= 0 else ''
    print(f'  {k:<12} {sign}{v:.4f}  {bar}')

## Part 9: News Decomposition Waterfall Chart

In [ ]:
def plot_news_waterfall(news: Dict[str, float], prev_fc: float, curr_fc: float) -> None:
    if not news:
        print('No news to plot')
        return

    items = sorted(news.items(), key=lambda x: abs(x[1]), reverse=True)
    labels = [k for k, _ in items]
    values = [v for _, v in items]
    colors = ['steelblue' if v >= 0 else 'firebrick' for v in values]

    running = prev_fc
    lefts = []
    for v in values:
        lefts.append(running if v >= 0 else running + v)
        running += v

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(
        range(len(labels)), values,
        left=lefts, color=colors, edgecolor='white', height=0.55
    )

    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=10)
    ax.axvline(prev_fc, color='gray', linewidth=1.5, linestyle='--',
               label=f'Previous: {prev_fc:.3f}')
    ax.axvline(curr_fc, color='darkgreen', linewidth=1.5, linestyle='-',
               label=f'Current: {curr_fc:.3f}')

    ax.set_xlabel('Contribution to nowcast revision')
    ax.set_title('News Decomposition: Nov 4 → Nov 15', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()


if len(evolution_records) >= 4:
    prev_fc = evolution_records[2].point_forecast  # Nov 4
    curr_fc = evolution_records[3].point_forecast  # Nov 15
    plot_news_waterfall(news, prev_fc, curr_fc)
else:
    print('Fewer than 4 records available; skipping waterfall')

## Summary

In this notebook you built a complete nowcasting pipeline:

| Step | Component | Key takeaway |
|------|-----------|-------------|
| Data generation | Synthetic factor model | Realistic correlations, revision noise |
| Vintage database | SQLite + `get_as_of()` | Immutable rows, pseudo-real-time query |
| Feature engineering | Ragged-edge fill + MIDAS matrix | Carry-forward default |
| Estimation | ElasticNetCV + TimeSeriesSplit | Scale on train only |
| Output | ForecastRecord dataclass | Fully auditable |
| Evaluation | Pseudo-real-time OOS | Compare to AR(1) benchmark |
| Interpretation | News decomposition waterfall | Coefficient × feature change |

Proceed to Notebook 02 to build the monitoring dashboard on top of this pipeline.